In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [184]:
df =  pd.read_json('orders.json')
df

,order_id,customer,amount,status
0,O1,Alice,1200,delivered
1,O2,Bob,-500,returned
2,O3,Charlie,abc,delivered
3,O4,David,2500,pending
4,O5,,1800,delivered
5,O6,Eve,,delivered


In [185]:
df.info()

valid = df

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   order_id  6 non-null      str  
 1   customer  6 non-null      str  
 2   amount    6 non-null      str  
 3   status    6 non-null      str  
dtypes: str(4)
memory usage: 324.0 bytes


In [ ]:
valid = valid[valid['amount'].notna()]
valid = valid[valid['amount'].apply(lambda x: str(x).isdigit())]


In [187]:
valid

,order_id,customer,amount,status
0,O1,Alice,1200,delivered
3,O4,David,2500,pending
4,O5,,1800,delivered


In [188]:
valid = valid.astype({'amount': 'int32'})
valid.info()

<class 'pandas.DataFrame'>
Index: 3 entries, 0 to 4
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   order_id  3 non-null      str  
 1   customer  3 non-null      str  
 2   amount    3 non-null      int32
 3   status    3 non-null      str  
dtypes: int32(1), str(3)
memory usage: 108.0 bytes


In [189]:
valid = valid[valid['customer'] != ""]
valid

,order_id,customer,amount,status
0,O1,Alice,1200,delivered
3,O4,David,2500,pending


In [190]:
revenue = df 
revenue = revenue[revenue['amount'].notna()]
revenue = revenue[revenue['amount'].apply(lambda x: str(x).lstrip('-').isdigit())]
revenue['amount'] = revenue['amount'].astype('int64')

In [191]:
revenue['amount'] = revenue['amount'].astype('int64')

In [192]:
revenue  = revenue.groupby('status')['amount'].sum().reset_index()
revenue

,status,amount
0,delivered,3000
1,pending,2500
2,returned,-500


In [193]:
top_customers = valid.groupby('customer')['amount'].sum().reset_index()
top_customers = top_customers.nlargest(1, 'amount')
top_customers

,customer,amount
1,David,2500


In [194]:
def error(df):
    df['error'] = df['amount'].apply(lambda x: "Invalid amount" if str(x).isalpha() else None)
    df['error'] = df['error'].fillna(df['amount'].apply(lambda x: "Missing amount" if pd.isna(x) or x == "" else None))
    df['error'] = df['error'].fillna(df['customer'].apply(lambda x: "Missing customer" if pd.isna(x) or x == "" else None))
    return df['error']

In [195]:
df['error']  = error(df)
df

,order_id,customer,amount,status,error
0,O1,Alice,1200,delivered,NaN
1,O2,Bob,-500,returned,NaN
2,O3,Charlie,abc,delivered,Invalid amount
3,O4,David,2500,pending,NaN
4,O5,,1800,delivered,Missing customer
5,O6,Eve,,delivered,Missing amount


In [196]:
error_log =  df[df['error'].notna()][['order_id', 'error']]
error_log

,order_id,error
2,O3,Invalid amount
4,O5,Missing customer
5,O6,Missing amount


In [197]:
print("VALID ORDERS:", len(valid))
print()
print('TOTAL REVENUE:', revenue[revenue['status'] == 'delivered']['amount'].sum())
print("TOTAL RETURNS:", revenue[revenue['status'] == 'returned']['amount'].sum())
print()
print("TOP CUSTOMER:", top_customers['customer'].values[0])
print()
print("ERROR LOG:")
for _, row in error_log.iterrows():
    print("Order", row['order_id'], '->', row['error'])

VALID ORDERS: 2

TOTAL REVENUE: 3000
TOTAL RETURNS: -500

TOP CUSTOMER: David

ERROR LOG:
Order O3 -> Invalid amount
Order O5 -> Missing customer
Order O6 -> Missing amount
